In [5]:
import sys
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import FastEmbedEmbeddings

# 1. Project Root Directory Setup
ROOT_DIR = Path.cwd()
# If cwd is inside RAG_src/processing, move up to project root
if ROOT_DIR.name in ["processing", "vectorDB", "RAG_src"]:
    ROOT_DIR = Path("D:/Projects/LUX/LUX_Data_Operations_Guide")

DOCS_DIR = ROOT_DIR / "docs"
CHROMA_PATH = ROOT_DIR / "RAG_src" / "vectorDB" / "chroma_db"

# 2. Embedding & Store Setup
def get_embedding_function():
    return FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 3. Loader & Chunking Function
def load_and_chunk_docs(docs_dir=DOCS_DIR):
    loader = DirectoryLoader(
        str(docs_dir),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    raw_documents = loader.load()

    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    all_chunks = []
    for doc in raw_documents:
        chunks = markdown_splitter.split_text(doc.page_content)
        for chunk in chunks:
            chunk.metadata["source"] = doc.metadata.get("source", "")
            all_chunks.append(chunk)

    return raw_documents, all_chunks

# 4. Ingestion Runner
def run_ingestion():
    print(f"Loading docs from: {DOCS_DIR}")
    raw_docs, chunks = load_and_chunk_docs()
    
    print(f"Creating vector store at: {CHROMA_PATH}")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=get_embedding_function(),
        persist_directory=str(CHROMA_PATH)
    )
    print(f"Successfully processed {len(raw_docs)} MD files into {len(chunks)} chunks!")

run_ingestion()

Loading docs from: D:\Projects\LUX\LUX_Data_Operations_Guide\docs
Creating vector store at: D:\Projects\LUX\LUX_Data_Operations_Guide\RAG_src\vectorDB\chroma_db
Successfully processed 19 MD files into 6 chunks!
